# MDP - YOLO26 Training (Local / Offline)

Trains an [Ultralytics YOLO26](https://docs.ultralytics.com/models/yolo26/) detector on **your own machine** (NVIDIA GPU / CUDA) using the `ultralytics` package. No Google Drive or Colab required.

YOLO26 is trained through the `ultralytics` package (`from ultralytics import YOLO`) - there is no separate training repo to clone. The only network access this workflow needs is a **one-time** download of the pretrained COCO starting checkpoint (`yolo26s.pt`) and the initial `pip install`.

# Step 1: Install & Verify Environment

Run the following once in a terminal (from the repo root or anywhere):

```bash
python -m venv .venv
source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install --upgrade pip
pip install ultralytics
```

If PyTorch is not already CUDA-enabled, install the CUDA build that matches your driver, e.g. for CUDA 12.1:

```bash
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
```

Then run the cell below to confirm the GPU is visible.

In [ ]:
# %pip install ultralytics   # only if you are NOT using a venv

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import ultralytics
ultralytics.checks()

# Step 2: Point to Your Dataset

Ultralytics uses the same YOLO dataset format as the old v5/v8 notebooks (`images/`, `labels/`, `data.yaml`), so a Roboflow export for v5/v8 works unchanged. Set the path to your dataset's `data.yaml` below. The `train`/`val`/`test` paths inside the yaml are resolved relative to the yaml file itself.

In [ ]:
from pathlib import Path

# Absolute or repo-relative path to your dataset's data.yaml
DATA_YAML = str(Path("Week8/datav5.yaml").resolve())   # <- change this to your dataset

# Where Ultralytics writes checkpoints (relative to this notebook's working dir)
PROJECT = "runs"

print("data.yaml:", DATA_YAML)
print("exists:", Path(DATA_YAML).exists())

# Step 3: Load the Pretrained Starting Point

The first run downloads the chosen checkpoint from the Ultralytics GitHub release and keeps it locally, so later runs are fully offline. Sizes: `yolo26n.pt` `yolo26s.pt` `yolo26m.pt` `yolo26l.pt` `yolo26x.pt`.

In [ ]:
from ultralytics import YOLO

# Pick the biggest size your GPU allows
model = YOLO("yolo26s.pt")

In [ ]:
!nvidia-smi

# Step 4: Train

- **epochs**: ~20-100 is plenty for this symbol task.
- **batch**: start with `-1` (auto-picks the largest batch your GPU fits), then set a fixed value such as `128` if you want reproducibility.
- **imgsz**: `416` (as used in the previous notebooks) or `640`.
- **fliplr=0.0** is important - horizontal flips would confuse the Left/Right arrow symbols.
- **resume**: if a run is interrupted, load `runs/train/weights/last.pt` and call `model.train(resume=True)`.
- Checkpoints are written to `<PROJECT>/train/weights/` as `best.pt` (best validation) and `last.pt` (final epoch).

In [ ]:
model.train(data=DATA_YAML, epochs=100, imgsz=416, batch=-1,
            fliplr=0.0, project=PROJECT, name="train", device=0)

# Step 5: Validate & Test the Trained Model

In [ ]:
# Evaluate the best checkpoint on the validation split
best = YOLO(f"{PROJECT}/train/weights/best.pt")
best.val(data=DATA_YAML)

# Run detection on your test images and save the annotated results
# TEST_SOURCE = str(Path("Week8/test/images").resolve())  # <- point at your test folder
best.predict(source="Week8/test/images", conf=0.25, save=True)

# Step 6: Deploy to the Inference Server

Copies `best.pt` into the inference server directory as the checkpoint that `load_model()` in `model.py` expects. The model.py path is `YOLO26_Week_9.pt` (Task 2 / left-right-bullseye); for Task 1 use `YOLO26_Week_8.pt` and update the path in `model.py`.

In [ ]:
import shutil
from pathlib import Path

# The inference server folder is 2 levels up from this notebook
server_dir = Path.cwd().resolve().parents[1] / "YOLOv5 Inference Server"
dest = server_dir / "YOLO26_Week_9.pt"   # use YOLO26_Week_8.pt for Task 1
shutil.copy(f"{PROJECT}/train/weights/best.pt", dest)
print("Copied best.pt to", dest)

## Notes

- **Offline caveat**: `yolo26s.pt` is fetched once over the network on first run (it is cached next to your script/notebook). Everything after that - training, validation, inference - is fully local.
- Runs are saved under `runs/` next to this notebook. Delete old `runs/train/*` folders or change `name=` to avoid collisions.
- The vendored YOLOv5 files in the inference server folder are unused legacy code; training here only needs the `ultralytics` package.